# BERT Sentiment Analysis on IMDB — with Attention Visualization

This notebook:
1. Fine-tunes `bert-base-cased` on IMDB reviews
2. Visualizes attention patterns for **both** the base (unfine-tuned) and fine-tuned BERT models

---

## 0. Environment Setup

In [ ]:
import os

# Detect runtime
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"

if IN_COLAB:
    %pip install -q circuitsvis bertviz
    # Node.js is needed by circuitsvis
    !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash - && sudo apt-get install -y nodejs
else:
    # In a local env, install manually if not already present:
    # pip install circuitsvis bertviz
    pass

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

import torch
from torch.utils.data import DataLoader
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

import transformers
print("Transformers:", transformers.__version__)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from evaluate import load as load_metric

# Attention visualisation
import circuitsvis as cv
from bertviz import head_view, model_view

## 2. Load & Preprocess IMDB Dataset

In [ ]:
dataset = load_dataset("imdb")

split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label",
)

train_dataset = split_dataset["train"]
val_dataset   = split_dataset["test"]
test_dataset  = dataset["test"]

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
MODEL_NAME = "bert-base-cased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    texts = [t.replace("<br /><br />", " ") for t in examples["text"]]
    return tokenizer(texts, truncation=True, max_length=256)

train_tokenized = train_dataset.map(preprocess_function, batched=True)
val_tokenized   = val_dataset.map(preprocess_function,   batched=True)
test_tokenized  = test_dataset.map(preprocess_function,  batched=True)

for ds in (train_tokenized, val_tokenized, test_tokenized):
    ds.rename_column_("label", "labels")
    ds.set_format("torch")

## 3. TF-IDF Baseline

In [ ]:
def clean_text(texts):
    return [t.replace("<br /><br />", " ") for t in texts]

train_texts  = clean_text(dataset["train"]["text"])
test_texts   = clean_text(dataset["test"]["text"])
train_labels = dataset["train"]["label"]
test_labels  = dataset["test"]["label"]

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)

clf = LogisticRegression(C=1.0, max_iter=1000)
clf.fit(X_train, train_labels)
preds = clf.predict(X_test)

print("[TF-IDF Baseline]")
print("Accuracy:", accuracy_score(test_labels, preds))
print("F1:      ", f1_score(test_labels, preds))
print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

## 4. Attention Visualization — Base BERT (Before Fine-Tuning)

We visualise attention **before** any fine-tuning so we can compare with the fine-tuned model later.

Two complementary tools are used:
* **circuitsvis** — compact, interactive head-pattern grids  
* **bertviz** — full head-view and model-view for BERT-style models

In [ ]:
# ── Helper: extract attention weights from any BertForSequenceClassification ──

def get_bert_attention(
    bert_model,
    bert_tokenizer,
    text: str,
):
    """
    Run a forward pass with output_attentions=True.

    Returns
    -------
    tokens      : list[str]  — string tokens (including [CLS] / [SEP])
    attentions  : tuple of tensors, one per layer,
                  each shaped (1, n_heads, seq_len, seq_len)
    inputs      : the tokenizer output (for bertviz)
    """
    clean = text.replace("<br /><br />", " ")
    inputs = bert_tokenizer(
        clean, return_tensors="pt", truncation=True, max_length=256
    ).to(device)

    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**inputs, output_attentions=True)

    # attentions: tuple of (1, n_heads, seq_len, seq_len), one per layer
    attentions = outputs.attentions
    token_ids  = inputs["input_ids"][0].tolist()
    tokens     = bert_tokenizer.convert_ids_to_tokens(token_ids)
    return tokens, attentions, inputs


# ── Helper: circuitsvis attention grid for a single layer ──

def visualize_bert_layer_circuitsvis(bert_model, bert_tokenizer, text: str, layer_id: int):
    """
    Show an interactive head-pattern grid (via circuitsvis) for one encoder layer.
    """
    tokens, attentions, _ = get_bert_attention(bert_model, bert_tokenizer, text)

    # shape: (n_heads, seq_len, seq_len)  — drop batch dim
    attn_layer = attentions[layer_id][0]  # (n_heads, seq_len, seq_len)

    print(f"Layer {layer_id} — {attn_layer.shape[0]} heads | tokens: {tokens}")
    display(
        cv.attention.attention_patterns(tokens=tokens, attention=attn_layer)
    )


# ── Helper: bertviz head-view (all layers at once) ──

def visualize_bert_bertviz(bert_model, bert_tokenizer, text: str):
    """
    Interactive bertviz head_view showing all layers simultaneously.
    """
    tokens, attentions, inputs = get_bert_attention(bert_model, bert_tokenizer, text)
    # bertviz expects a list of tensors with the batch dim present
    head_view(attentions, tokens)

In [ ]:
# Load the BASE model (no fine-tuning) with output_attentions support
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)
base_model.eval()
print("Base BERT loaded.")

In [ ]:
# ── Pick a sample review for attention inspection ──
SAMPLE_POS = "This film was an absolute delight. The acting was superb and the story kept me hooked throughout."
SAMPLE_NEG = "Terrible movie. The plot made no sense and the acting was wooden and unconvincing."

# circuitsvis — layer 0
print("=== BASE BERT | Positive review | Layer 0 ===")
visualize_bert_layer_circuitsvis(base_model, tokenizer, SAMPLE_POS, layer_id=0)

In [ ]:
print("=== BASE BERT | Negative review | Layer 0 ===")
visualize_bert_layer_circuitsvis(base_model, tokenizer, SAMPLE_NEG, layer_id=0)

In [ ]:
# bertviz full head-view — BASE model
print("=== BASE BERT | bertviz head_view | Positive review ===")
visualize_bert_bertviz(base_model, tokenizer, SAMPLE_POS)

## 5. Fine-Tune BERT on IMDB

In [ ]:
import wandb
wandb.login()

# ── W&B config ────────────────────────────────────────────────────────────────
WANDB_ENTITY  = "Mini-project-AML-2026"   # your W&B entity / team name
WANDB_PROJECT = "test2"
WANDB_RUN_NAME = "bert-base-cased-v03"
ARTIFACT_NAME  = "bert-imdb-finetuned"    # name used when saving/loading the model artifact
# ─────────────────────────────────────────────────────────────────────────────

wandb_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model_name": MODEL_NAME,
        "num_labels": 2,
        "max_length": 256,
    },
)

In [ ]:
# We keep base_model separate for attention comparison.
# The Trainer will fine-tune this independent copy.
ft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

training_args = TrainingArguments(
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.1,
    save_total_limit=2,
    logging_steps=100,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    dataloader_pin_memory=torch.cuda.is_available(),
    output_dir="./results",
    report_to="wandb",
)

metric = load_metric("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from transformers.integrations import WandbCallback
trainer.remove_callback(WandbCallback)
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_tokenized)
logits = predictions.predictions
preds  = logits.argmax(axis=1)
labels = predictions.label_ids

print("[Fine-tuned BERT]")
print("Accuracy:", accuracy_score(labels, preds))
print("F1:      ", f1_score(labels, preds))
print(classification_report(labels, preds))
print(confusion_matrix(labels, preds))

### 5a. Save Fine-Tuned Model as a W&B Artifact

This saves the best checkpoint (weights + tokenizer config) to W&B Artifacts so it  
survives the Colab session. The artifact is versioned automatically — every re-run  
that calls this cell creates a new `v1`, `v2`, … while keeping all previous versions.

In [ ]:
import os

SAVE_DIR = "./best_model"

# Save model weights + tokenizer into a local directory first
trainer.save_model(SAVE_DIR)          # saves the best checkpoint loaded by load_best_model_at_end
tokenizer.save_pretrained(SAVE_DIR)   # so the artifact is fully self-contained

# Log as a W&B Artifact attached to the current run
artifact = wandb.Artifact(
    name=ARTIFACT_NAME,
    type="model",
    description=f"Fine-tuned {MODEL_NAME} on IMDB (best val-F1 checkpoint)",
    metadata={
        "model_name": MODEL_NAME,
        "num_labels": 2,
        "max_length": 256,
    },
)
artifact.add_dir(SAVE_DIR)            # upload every file in SAVE_DIR
wandb_run.log_artifact(artifact)

print(f"Model saved to W&B Artifact '{ARTIFACT_NAME}' under run '{WANDB_RUN_NAME}'.")
print("You can browse it at: https://wandb.ai/", WANDB_ENTITY, "/", WANDB_PROJECT, "/artifacts")

### 5b. Reload Fine-Tuned Model from W&B Artifact (new session)

Run **only this section** when you open a fresh Colab session and want to skip  
re-training. It downloads the artifact and loads the model directly into `ft_model`.

> **Tip:** `latest` always resolves to the most recently logged version.  
> Pin to a specific version like `v2` if you need reproducibility.

In [ ]:
# ── Set to True when restoring from a previous session; False when training fresh ──
LOAD_FROM_ARTIFACT = False
ARTIFACT_VERSION   = "latest"   # e.g. "v0", "v1", or "latest"
# ──────────────────────────────────────────────────────────────────────────────

if LOAD_FROM_ARTIFACT:
    import wandb
    wandb.login()

    # Start a lightweight run just for artifact download (no training logged)
    restore_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        job_type="inference",
        name="restore-for-inference",
    )

    artifact_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{ARTIFACT_NAME}:{ARTIFACT_VERSION}"
    artifact = restore_run.use_artifact(artifact_path, type="model")
    download_dir = artifact.download()          # downloads to ./artifacts/<name>:<version>/
    print(f"Artifact downloaded to: {download_dir}")

    # Load model and tokenizer from the downloaded directory
    ft_model  = AutoModelForSequenceClassification.from_pretrained(download_dir).to(device)
    tokenizer = BertTokenizer.from_pretrained(download_dir)
    ft_model.eval()

    restore_run.finish()
    print("Fine-tuned model loaded from W&B Artifact — ready for inference and attention analysis.")
else:
    print("LOAD_FROM_ARTIFACT=False — using ft_model from the training run above.")

## 6. Attention Visualization — Fine-Tuned BERT

Now we run the **same** helpers on `ft_model` (the Trainer's best checkpoint)  
and compare the patterns side-by-side with the base model.

In [ ]:
# Make sure ft_model is in eval mode after training
ft_model.eval()

# circuitsvis — fine-tuned, layer 0
print("=== FINE-TUNED BERT | Positive review | Layer 0 ===")
visualize_bert_layer_circuitsvis(ft_model, tokenizer, SAMPLE_POS, layer_id=0)

In [ ]:
print("=== FINE-TUNED BERT | Negative review | Layer 0 ===")
visualize_bert_layer_circuitsvis(ft_model, tokenizer, SAMPLE_NEG, layer_id=0)

In [ ]:
# bertviz head-view — fine-tuned model
print("=== FINE-TUNED BERT | bertviz head_view | Positive review ===")
visualize_bert_bertviz(ft_model, tokenizer, SAMPLE_POS)

## 7. Side-by-Side Comparison (Static Heatmaps)

For a quick, printable comparison we extract the mean attention over heads  
for a chosen layer and plot both models next to each other with matplotlib.

In [ ]:
def mean_head_attention(bert_model, bert_tokenizer, text: str, layer_id: int):
    """
    Return the mean-over-heads attention matrix for one layer.
    Shape: (seq_len, seq_len)  —  values in [0, 1].
    """
    tokens, attentions, _ = get_bert_attention(bert_model, bert_tokenizer, text)
    # attentions[layer_id] : (1, n_heads, seq_len, seq_len)
    attn = attentions[layer_id][0]          # (n_heads, seq_len, seq_len)
    mean_attn = attn.mean(dim=0).cpu().numpy()  # (seq_len, seq_len)
    return tokens, mean_attn


def plot_side_by_side(text: str, layer_id: int = 0):
    tokens_b, attn_b = mean_head_attention(base_model, tokenizer, text, layer_id)
    tokens_f, attn_f = mean_head_attention(ft_model,   tokenizer, text, layer_id)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, attn, title in zip(
        axes,
        [attn_b, attn_f],
        [f"Base BERT — Layer {layer_id}", f"Fine-tuned BERT — Layer {layer_id}"],
    ):
        im = ax.imshow(attn, vmin=0, vmax=attn.max(), cmap="Blues")
        ax.set_xticks(range(len(tokens_b)))
        ax.set_yticks(range(len(tokens_b)))
        ax.set_xticklabels(tokens_b, rotation=90, fontsize=8)
        ax.set_yticklabels(tokens_b, fontsize=8)
        ax.set_title(title, fontsize=11)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    short = text[:60] + "..." if len(text) > 60 else text
    fig.suptitle(f'"{short}"', fontsize=10, style="italic")
    plt.tight_layout()
    plt.show()


# -- Positive review, layer 0
plot_side_by_side(SAMPLE_POS, layer_id=0)

In [ ]:
# -- Negative review, layer 0
plot_side_by_side(SAMPLE_NEG, layer_id=0)

In [ ]:
# Experiment: try a different layer (e.g. 6 — middle of the 12-layer encoder)
plot_side_by_side(SAMPLE_POS, layer_id=6)

## 8. [CLS] Token Attention (Sentiment-Relevant Heads)

BERT uses the `[CLS]` token for classification.  
Plotting the attention **from** `[CLS]` **to** all other tokens gives a  
direct readout of what the model attends to when making its prediction.

In [ ]:
def plot_cls_attention(text: str, layer_id: int = 11):
    """
    Bar chart of [CLS] attention weights (mean over heads) for base vs fine-tuned.
    """
    tokens_b, attn_b = mean_head_attention(base_model, tokenizer, text, layer_id)
    tokens_f, attn_f = mean_head_attention(ft_model,   tokenizer, text, layer_id)

    # Row 0 of the attention matrix = what [CLS] attends to
    cls_b = attn_b[0]
    cls_f = attn_f[0]

    x = np.arange(len(tokens_b))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(10, len(tokens_b) * 0.5), 4))
    ax.bar(x - width / 2, cls_b, width, label="Base BERT",        color="steelblue", alpha=0.8)
    ax.bar(x + width / 2, cls_f, width, label="Fine-tuned BERT",  color="coral",     alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(tokens_b, rotation=60, ha="right", fontsize=9)
    ax.set_ylabel("[CLS] attention weight (mean over heads)")
    ax.set_title(f"[CLS] Attention — Layer {layer_id}")
    ax.legend()
    plt.tight_layout()
    plt.show()


print("Positive review — last layer")
plot_cls_attention(SAMPLE_POS, layer_id=11)

print("Negative review — last layer")
plot_cls_attention(SAMPLE_NEG, layer_id=11)

## 9. Interactive Exploration

Use the cells below to examine **any** text and layer you like.

In [ ]:
# ── Edit these to explore ──────────────────────────────────────────────────────
CUSTOM_TEXT = "I wasn't sure about this film at first but it completely won me over by the end."
LAYER       = 0          # 0–11 for bert-base
# ──────────────────────────────────────────────────────────────────────────────

print("── circuitsvis: Base BERT ──")
visualize_bert_layer_circuitsvis(base_model, tokenizer, CUSTOM_TEXT, LAYER)

print("── circuitsvis: Fine-tuned BERT ──")
visualize_bert_layer_circuitsvis(ft_model, tokenizer, CUSTOM_TEXT, LAYER)

print("── Static heatmap comparison ──")
plot_side_by_side(CUSTOM_TEXT, layer_id=LAYER)

print("── [CLS] attention bar chart ──")
plot_cls_attention(CUSTOM_TEXT, layer_id=LAYER)